# Análisis Descriptivo y Estimación de Elasticidad Precio-Demanda

## 1. Configuración de rutas e importación de librerías

In [ ]:
library(dplyr)
library(ggplot2)
library(readr)
library(broom)
library(tidyr)

input_path <- "C:/Users/owoc9/Documents/U Javeriana/Laboratorio Analítica/DGO/Precios y Canales/Version Ajustada_20260223/supermarket_sales_nuevo.csv"
output_path <- "C:/Users/owoc9/Documents/U Javeriana/Laboratorio Analítica/DGO/Precios y Canales/Version_Ajustada_20260309/"

if(!dir.exists(output_path)) dir.create(output_path, recursive = TRUE)

## 2. Lectura y renombrado de variables

In [ ]:
data <- read.csv(input_path, sep = ",")

cat("\n--- Vista preliminar de los datos ---\n")
head(data)

## 3. Análisis descriptivo univariado

In [ ]:
cat("\n--- Resumen general de variables numéricas ---\n")
summary(select(data, unit_price, quantity, total, cogs, gross_income, rating))

cat("\n--- Distribución de variables categóricas ---\n")
categorical_vars <- c("branch", "city", "customer_type", "gender", "product_line", "payment")
for (var in categorical_vars) {
  cat(paste0("\nFrecuencias de ", var, ":\n"))
  print(table(data[[var]]))
}

In [ ]:
# Gráficos univariados
ggplot(data, aes(x = unit_price)) + 
  geom_histogram(binwidth = 5, fill = "steelblue", color = "white") +
  labs(title = "Distribución del Precio Unitario", x = "Precio Unitario", y = "Frecuencia")

ggplot(data, aes(x = quantity)) + 
  geom_histogram(binwidth = 1, fill = "orange", color = "red") +
  labs(title = "Distribución de Cantidad", x = "Cantidad", y = "Frecuencia")

## 4. Análisis bivariado: Precio vs Cantidad

In [ ]:
cat("\n--- Correlación entre precio y cantidad ---\n")
cor(data$unit_price, data$quantity)

ggplot(data, aes(x = quantity, y = unit_price)) +
  geom_point(alpha = 0.6) +
  labs(title = "Relación Precio - Cantidad", x = "Precio Unitario", y = "Cantidad Vendida")

## 5. Modelo de elasticidad precio-demanda por canal (Branch)

In [ ]:
# Transformaciones logarítmicas
data <- data %>%
  mutate(
    log_price = log(unit_price),
    log_quantity = log(quantity)
  )

regla_decision_elasticidad <- "
--- Regla de decisión (según |E_p|) ---
|E_p| = 0           -> Perfectamente inelástica
0 < |E_p| < 1       -> Inelástica
|E_p| = 1           -> Unitaria
|E_p| > 1           -> Elástica
E_p >= 0             -> No negativa o inusual
"

In [ ]:
# Estimación por canal (branch)
elasticity_models <- data %>%
  group_by(branch) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- tidy(m)
    glance_m <- glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por canal ---\n")
elasticity_models

write.csv(elasticity_models, file.path(output_path, "elasticidad_por_branch.csv"), row.names = FALSE)

## 6. Interpretación automática básica

In [ ]:
cat("\n--- Interpretación básica ---\n")
for (i in 1:nrow(elasticity_models)) {
  b <- elasticity_models$branch[i]
  e <- elasticity_models$elasticity[i]
  abs_e <- abs(e)
  msg <- ifelse(e >= 0,
                "Relación no negativa o inusual: precio y cantidad se mueven en la misma dirección.",
                ifelse(abs_e == 0,
                       "Demanda perfectamente inelástica: la cantidad no varía ante cambios en el precio.",
                       ifelse(abs_e > 0 & abs_e < 1,
                              "Demanda inelástica: la cantidad varía menos que proporcional al cambio en el precio.",
                              ifelse(abs_e == 1,
                                     "Demanda unitaria: la cantidad varía en la misma proporción que el precio.",
                                     "Demanda elástica: la cantidad varía más que proporcional al cambio en el precio."
                              ))))
  cat(paste0("Sucursal ", b, ": Elasticidad = ", round(e, 3), " -> ", msg, "\n"))
}

cat("\n--- Proceso finalizado. Resultados exportados y mostrados en consola. ---\n")

## 7. Comparación de elasticidades entre canales

### 7.1 Por canal y tipo de cliente

In [ ]:
elasticity_branch_customer <- data %>%
  group_by(branch, customer_type) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- tidy(m)
    glance_m <- glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por canal segmentados por tipo de cliente ---\n")
elasticity_branch_customer
cat(regla_decision_elasticidad)

### 7.2 Por canal y ciudad

In [ ]:
elasticity_branch_city <- data %>%
  group_by(branch, city) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- tidy(m)
    glance_m <- glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por canal segmentados por ciudad ---\n")
elasticity_branch_city
cat(regla_decision_elasticidad)

### 7.3 Por canal y género

In [ ]:
elasticity_branch_gender <- data %>%
  group_by(branch, gender) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- tidy(m)
    glance_m <- glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por canal segmentados por género ---\n")
elasticity_branch_gender
cat(regla_decision_elasticidad)

## 8. Visualización de resultados

In [ ]:
ggplot(data, aes(x = log_price, y = log_quantity)) +
  geom_point(alpha = 0.4) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(title = "Relación entre log(Quantity) y log(Price)", x = "log(Unit Price)", y = "log(Quantity)") +
  theme_minimal()

ggplot(elasticity_models, aes(x = branch, y = elasticity)) +
  geom_col() +
  coord_cartesian(ylim = c(-2.5, 0)) +
  geom_hline(yintercept = -1, linetype = "dashed", color = "red") +
  labs(title = "Comparación elasticidad por sucursal",
       x = "Sucursal",
       y = "Elasticidad Precio de la Demanda") +
  theme_minimal()

## 9. Impacto del 10% de aumento en el precio por sucursal

In [ ]:
val_promedio <- data %>%
  group_by(branch) %>%
  summarise(average_quantity = mean(quantity), average_price = mean(unit_price))

simulacion_010 <- elasticity_models %>%
  left_join(val_promedio, by = "branch") %>%
  mutate(price_change = 0.10, 
         quantity_change_010 = elasticity * price_change,
         new_quantity = average_quantity * (1 + quantity_change_010),
         new_price = average_price * (1 + price_change),
         revenue_initial = average_price * average_quantity,
         revenue_new = new_price * new_quantity,
         revenue_change = revenue_new - revenue_initial)

ggplot(simulacion_010, aes(x = reorder(branch, revenue_initial), y = revenue_initial)) +
  geom_col(fill = "blue") + 
  labs(title = "Ventas por sucursal antes del aumento del 10%", x = "Sucursal", y = "Ventas") +
  theme_minimal()

ggplot(simulacion_010, aes(x = reorder(branch, revenue_new), y = revenue_new)) +
  geom_col(fill = "red") +
  labs(title = "Ventas por Sucursal después del aumento del 10%", x = "Sucursal", y = "Ventas") +
  theme_minimal()

simulacion_010

## 10. Identificación del producto inelástico

In [ ]:
elasticity_products <- data %>%
  group_by(product_line) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- broom::tidy(m)
    glance_m <- broom::glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por línea de producto ---\n")
elasticity_products
cat(regla_decision_elasticidad)

In [ ]:
cat("\n--- Interpretación por productos ---\n")
for (i in 1:nrow(elasticity_products)) {
  b <- elasticity_products$product_line[i]
  e <- elasticity_products$elasticity[i]
  abs_e <- abs(e)
  msg <- ifelse(e >= 0,
                "Relación no negativa o inusual: precio y cantidad se mueven en la misma dirección.",
                ifelse(abs_e == 0,
                       "Demanda perfectamente inelástica: la cantidad no varía ante cambios en el precio.",
                       ifelse(abs_e > 0 & abs_e < 1,
                              "Demanda inelástica: la cantidad varía menos que proporcional al cambio en el precio.",
                              ifelse(abs_e == 1,
                                     "Demanda unitaria: la cantidad varía en la misma proporción que el precio.",
                                     "Demanda elástica: la cantidad varía más que proporcional al cambio en el precio."
                              ))))
  cat(paste0("Línea de producto ", b, ": Elasticidad = ", round(e, 3), " -> ", msg, "\n"))
}

elasticity_products <- elasticity_products %>% 
  mutate(abs_elasticity = abs(elasticity))

elasticity_products %>%
  arrange(abs_elasticity) %>%
  slice(1)

cat("Efectivamente la línea más inelástica es la que tiene menor |elasticidad|")

## 11. Ejemplo extra: Diferencia en cantidad entre Member y Normal

In [ ]:
data %>%
  group_by(customer_type) %>%
  summarise(avg_quantity = mean(quantity), avg_sales = mean(total))

ggplot(data, aes(x = customer_type, y = quantity)) +
  geom_boxplot() +
  theme_minimal() +
  labs(title = "Distribución de Cantidad por Tipo de Cliente",
       x = "Tipo de Cliente",
       y = "Cantidad")

## 12. Ejercicio 1: Elasticidades por customer_type y branch

In [ ]:
elasticity_ej1 <- data %>%
  group_by(customer_type, branch) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- broom::tidy(m)
    glance_m <- broom::glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por customer_type y branch ---\n")
elasticity_ej1
cat(regla_decision_elasticidad)

cat("\n--- Interpretación ---\n")
for (i in 1:nrow(elasticity_ej1)) {
  msg <- ifelse(elasticity_ej1$elasticity[i] >= 0,
                "Relación no negativa o inusual.",
                ifelse(abs(elasticity_ej1$elasticity[i]) < 1,
                       "Demanda inelástica.",
                       ifelse(abs(elasticity_ej1$elasticity[i]) == 1,
                              "Demanda unitaria.",
                              "Demanda elástica.")))
  cat(paste0(elasticity_ej1$customer_type[i], " - ", elasticity_ej1$branch[i], ": Elasticidad = ", 
            round(elasticity_ej1$elasticity[i], 3), " -> ", msg, "\n"))
}

## 13. Ejercicio 2: Elasticidades por branch y gender

In [ ]:
elasticity_ej2 <- data %>%
  group_by(branch, gender) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- broom::tidy(m)
    glance_m <- broom::glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por branch y gender ---\n")
elasticity_ej2
cat(regla_decision_elasticidad)

cat("\n--- Interpretación ---\n")
for (i in 1:nrow(elasticity_ej2)) {
  msg <- ifelse(elasticity_ej2$elasticity[i] >= 0,
                "Relación no negativa o inusual.",
                ifelse(abs(elasticity_ej2$elasticity[i]) < 1,
                       "Demanda inelástica.",
                       ifelse(abs(elasticity_ej2$elasticity[i]) == 1,
                              "Demanda unitaria.",
                              "Demanda elástica.")))
  cat(paste0(elasticity_ej2$branch[i], " - ", elasticity_ej2$gender[i], ": Elasticidad = ", 
            round(elasticity_ej2$elasticity[i], 3), " -> ", msg, "\n"))
}

## 14. Ejercicio 3: Elasticidades por product_line y método de pago

In [ ]:
# Calcule las elasticidades por product_line y payment. Interprete los resultados.
elasticity_ej3 <- data %>%
  group_by(product_line, payment) %>%
  do({
    m <- lm(log_quantity ~ log_price, data = .)
    tidy_m <- broom::tidy(m)
    glance_m <- broom::glance(m)
    tidy_m$r.squared <- glance_m$r.squared
    tidy_m
  }) %>%
  filter(term == "log_price") %>%
  rename(
    elasticity = estimate,
    r_squared = r.squared
  ) %>%
  mutate(
    tipo_elasticidad = dplyr::case_when(
      elasticity >= 0 ~ "No negativa o inusual",
      abs(elasticity) == 0 ~ "Perfectamente inelástica",
      abs(elasticity) > 0 & abs(elasticity) < 1 ~ "Inelástica",
      abs(elasticity) == 1 ~ "Unitaria",
      TRUE ~ "Elástica"
    )
  )

cat("\n--- Resultados de elasticidad por product_line y payment ---\n")
elasticity_ej3
cat(regla_decision_elasticidad)

cat("\n--- Interpretación ---\n")
for (i in 1:nrow(elasticity_ej3)) {
  msg <- ifelse(elasticity_ej3$elasticity[i] >= 0,
                "Relación no negativa o inusual.",
                ifelse(abs(elasticity_ej3$elasticity[i]) < 1,
                       "Demanda inelástica.",
                       ifelse(abs(elasticity_ej3$elasticity[i]) == 1,
                              "Demanda unitaria.",
                              "Demanda elástica.")))
  cat(paste0(elasticity_ej3$product_line[i], " - ", elasticity_ej3$payment[i], ": Elasticidad = ", 
            round(elasticity_ej3$elasticity[i], 3), " -> ", msg, "\n"))
}